# Video → 3D Gaussian Splat on a free cloud GPU

Runs the CS535 trainer (`trainer/`) on **Google Colab** or **Kaggle**: frames → COLMAP (pycolmap) →
optimization with the Vulkan renderer's forward/backward passes, sharing memory with PyTorch on CUDA.
The result is a model folder in the reference 3DGS layout. Download it, put it in `data/models/` and open it
in `splat_lab` (File > Load Scene).

**Before you start**
- Colab: *Runtime > Change runtime type > T4 GPU*. Kaggle: *Settings > Accelerator > GPU T4*, and turn on *Internet*.
- The renderer needs the NVIDIA **Vulkan** driver. Step 2 checks for it. If it fails, this runtime cannot run
  the trainer (use the Dockerfile on a host where you can run Docker with the NVIDIA Container Toolkit).
- A T4 is about 2–3× slower than a desktop RTX 30-series card. A full 30k-iteration run takes on the order
  of 1–2 hours, so for a first try use fewer iterations (below). Colab's free tier may disconnect long sessions.

## 1. Settings

In [ ]:
import os, sys

# Where to get the code: a git URL (cloned with submodules), or a zip of the repository uploaded to the
# working directory (set REPO_URL = "" and ZIP_NAME to its file name).
REPO_URL = "https://github.com/<you>/<repo>.git"
ZIP_NAME = "repo.zip"

# The video to train on. Leave empty to upload one in step 4 (Colab), or point it at a dataset file
# (Kaggle: /kaggle/input/<dataset>/<video>.mp4).
VIDEO = ""

ITERATIONS = 7000          # 30000 = reference schedule (best quality, slowest)
FRAMES = 150               # frames extracted from the video
MAX_SIZE = 1600            # longer image side in pixels
MAX_GAUSSIANS = 3_000_000  # cap (0 = unlimited); lower it if the GPU runs out of memory

# Progress previews: every PREVIEW_EVERY iterations the trainer saves the current render next to the real
# frame (camera PREVIEW_CAMERA of cameras.json; 0 is a held-out view). They are shown below the training
# cell as they are made, and copied to the share folder.
PREVIEW_EVERY = 500        # 0 = off
PREVIEW_CAMERA = 0

# Sharing: previews, a timelapse GIF and the final model zip are copied here as they are produced.
# Colab: SHARE_TO_DRIVE saves to Google Drive (share the folder from Drive). Kaggle: /kaggle/working is
# kept as the notebook's output (share or publish it from the Output panel).
SHARE_TO_DRIVE = True
RUN_NAME = "splat_run"

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle")
WORK = "/kaggle/working" if IN_KAGGLE else "/content"
REPO = f"{WORK}/splat"
OUT = f"{WORK}/out"
print("Colab" if IN_COLAB else "Kaggle" if IN_KAGGLE else "other environment", "- working directory", WORK)

SHARE_DIR = f"{WORK}/share/{RUN_NAME}"
if IN_COLAB and SHARE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    SHARE_DIR = f"/content/drive/MyDrive/splat_runs/{RUN_NAME}"
os.makedirs(f"{SHARE_DIR}/previews", exist_ok=True)
print("Sharing to", SHARE_DIR)

## 2. GPU and Vulkan

In [ ]:
import subprocess, textwrap
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
                     capture_output=True, text=True).stdout)

# Vulkan loader + tools, shader compiler, build tools.
!apt-get -qq update > /dev/null && apt-get -qq install -y cmake ninja-build glslang-tools libvulkan1 vulkan-tools zip > /dev/null

# Point the Vulkan loader at NVIDIA's driver (the container has the driver libraries but no ICD manifest).
os.makedirs("/usr/share/vulkan/icd.d", exist_ok=True)
with open("/usr/share/vulkan/icd.d/nvidia_icd.json", "w") as f:
    f.write('{"file_format_version":"1.0.0","ICD":{"library_path":"libGLX_nvidia.so.0","api_version":"1.3"}}\n')

info = subprocess.run(["vulkaninfo", "--summary"], capture_output=True, text=True)
devices = [l.strip() for l in info.stdout.splitlines() if "deviceName" in l]
print("\n".join(devices) or info.stderr[-2000:])
if not any("NVIDIA" in d or "Tesla" in d for d in devices):
    raise SystemExit("No NVIDIA Vulkan device: this runtime does not expose NVIDIA's Vulkan driver, "
                     "so the trainer cannot run here.")
print("Vulkan OK")

## 3. Code and build

Only the core renderer and the `splat_vk` Python module are built (`-DSPLAT_BUILD_VIEWER=OFF`, no window system needed).

In [ ]:
if not os.path.exists(REPO):
    if REPO_URL and "<you>" not in REPO_URL:
        !git clone -q --recursive {REPO_URL} {REPO}
    elif os.path.exists(f"{WORK}/{ZIP_NAME}"):
        import shutil, zipfile
        tmp = f"{WORK}/splat_unzip"
        zipfile.ZipFile(f"{WORK}/{ZIP_NAME}").extractall(tmp)
        top = os.listdir(tmp)
        # A zip of the repository folder has one top-level directory; use what is inside it.
        shutil.move(f"{tmp}/{top[0]}" if len(top) == 1 and os.path.isdir(f"{tmp}/{top[0]}") else tmp, REPO)
    else:
        raise SystemExit("Set REPO_URL to your repository, or upload the repository zip as ZIP_NAME.")
!cd {REPO} && git submodule update --init --recursive -q 2>/dev/null; true

!cmake -S {REPO} -B {REPO}/build -G Ninja -DCMAKE_BUILD_TYPE=Release -DSPLAT_BUILD_VIEWER=OFF -DPython_EXECUTABLE={sys.executable} > /dev/null
!cmake --build {REPO}/build --target splat_vk 2>&1 | tail -3
!ls {REPO}/build/splat_vk*

## 4. Video

In [ ]:
if not VIDEO:
    if IN_COLAB:
        from google.colab import files
        uploaded = files.upload()  # pick the video file; it is saved in the current directory
        VIDEO = os.path.abspath(next(iter(uploaded)))
    else:
        raise SystemExit("Set VIDEO to the video's path (Kaggle: add it as a dataset under /kaggle/input).")
assert os.path.exists(VIDEO), VIDEO
print(VIDEO, f"{os.path.getsize(VIDEO) / 2**20:.1f} MiB")

## 5. Train

`bootstrap.py` installs whatever is missing (pycolmap, OpenCV; PyTorch is preinstalled here) and runs `train.py`. Progress is printed as it goes (COLMAP's own log lines are hidden), and each preview image appears below as it is saved and is copied to the share folder.

In [ ]:
import json, shutil, subprocess, time
from IPython.display import Image as IPImage, display

env = dict(os.environ, SPLAT_NO_VENV="1", PYTHONUNBUFFERED="1")
cmd = [sys.executable, "-u", f"{REPO}/trainer/bootstrap.py", VIDEO, "-o", OUT,
       "--iterations", str(ITERATIONS), "--frames", str(FRAMES), "--max-size", str(MAX_SIZE),
       "--max-gaussians", str(MAX_GAUSSIANS), "--snapshot-every", "0", "--module-dir", f"{REPO}/build",
       "--preview-every", str(PREVIEW_EVERY), "--preview-camera", str(PREVIEW_CAMERA), "--progress-json"]
start = time.time()
previews = []
with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env) as p:
    for line in p.stdout:
        if line.startswith("@@"):
            event = json.loads(line[2:])
            if event["type"] == "preview":
                previews.append(event["path"])
                shutil.copy(event["path"], f"{SHARE_DIR}/previews/")
                print(f"--- preview, iteration {event['iteration']} (PSNR {event['psnr']:.2f} dB)")
                display(IPImage(filename=event["path"], width=900))
            elif event["type"] == "error":
                print("ERROR:", event["message"])
            continue
        if line.startswith(("I20", "W20")):  # COLMAP info/warning lines
            continue
        print(line, end="")
print(f"exit code {p.returncode}, {(time.time() - start) / 60:.1f} min")
for name in ("cameras.json", "test_cameras.json", "cfg_args"):
    if os.path.exists(f"{OUT}/{name}"):
        shutil.copy(f"{OUT}/{name}", SHARE_DIR)

## 6. Preview a held-out view

Renders the trained model at the first test camera with the Vulkan renderer, next to the real frame.

In [ ]:
import glob, json, re
import numpy as np, torch
import matplotlib.pyplot as plt
from PIL import Image
sys.path.insert(0, f"{REPO}/trainer")
from splat_trainer import vk

ply = max(glob.glob(f"{OUT}/point_cloud/iteration_*/point_cloud.ply"),
          key=lambda p: int(re.search(r"iteration_(\d+)", p).group(1)))
raw = open(ply, "rb").read()
end = raw.index(b"end_header\n") + len(b"end_header\n")
n = int(re.search(rb"element vertex (\d+)", raw[:end]).group(1))
a = np.frombuffer(raw[end:], "<f4").reshape(n, 62)
fields = {"means": a[:, 0:3], "sh_dc": a[:, 6:9],
          "sh_rest": a[:, 9:54].reshape(n, 3, 15).transpose(0, 2, 1).reshape(n, 45),
          "opacities": a[:, 54:55], "scales": a[:, 55:58], "rotations": a[:, 58:62]}

rast = vk.VulkanRasterizer(module_dir=f"{REPO}/build")
rast.ensure_capacity(n)
for k, v in fields.items():
    rast.params[k][:n].copy_(torch.from_numpy(np.ascontiguousarray(v)).cuda())

c = json.load(open(f"{OUT}/test_cameras.json"))[0]
c2w = np.eye(4)
c2w[:3, :3], c2w[:3, 3] = np.array(c["rotation"]), np.array(c["position"])
cam = vk.Camera(c["img_name"], c["width"], c["height"], c["fx"], c["fy"], c["width"] / 2, c["height"] / 2,
                np.linalg.inv(c2w))
with torch.no_grad():
    img = rast.render_raw(n, 3, cam).clamp(0, 1).permute(1, 2, 0).cpu().numpy()
photo = glob.glob(f"{OUT}/source/images/{c['img_name']}.*")
fig, ax = plt.subplots(1, 2 if photo else 1, figsize=(16, 6), squeeze=False)
ax[0, 0].imshow(img); ax[0, 0].set_title(f"Ours ({n:,} Gaussians)")
if photo:
    ax[0, 1].imshow(Image.open(photo[0])); ax[0, 1].set_title(f"Held-out frame {c['img_name']}")
for x in ax.flat: x.axis("off")
plt.show()

## 7. Timelapse

All previews as an animated GIF (saved to the share folder).

In [ ]:
import glob
from PIL import Image
frames = sorted(glob.glob(f"{OUT}/previews/iter_*.jpg"))
if frames:
    imgs = [Image.open(f).convert("RGB") for f in frames]
    w = 960
    imgs = [im.resize((w, round(im.height * w / im.width))) for im in imgs]
    gif = f"{SHARE_DIR}/timelapse.gif"
    imgs[0].save(gif, save_all=True, append_images=imgs[1:] + [imgs[-1]] * 3, duration=600, loop=0)
    print(gif, len(frames), "frames")
    display(IPImage(filename=gif, width=900))
else:
    print("No previews (PREVIEW_EVERY = 0).")

## 8. Download

The zip has the model (`point_cloud/`, `cameras.json`, `test_cameras.json`, `cfg_args`, `input.ply`) without the extracted frames. Unzip it into `data/models/<name>/` and open it with File > Load Scene.

In [ ]:
zip_path = f"{WORK}/splat_model.zip"
!cd {OUT} && rm -f {zip_path} && zip -qr {zip_path} . -x "source/*" "progress/*" "work/*"
print(zip_path, f"{os.path.getsize(zip_path) / 2**20:.0f} MiB")
shutil.copy(zip_path, SHARE_DIR)
print("Copied to", SHARE_DIR)
if IN_COLAB:
    from google.colab import files
    files.download(zip_path)
else:
    print("Kaggle: download it from the Output panel (it is in /kaggle/working).")